# Deteccion anomalias insfraestructura ferroviaria.

Version basica de deteccion de anomalias autoencoder, se ira mejorando para el TFM, por ahora para proyecto MLOps nos vale para desarrollar todas las buenas practicas mas que un modelo brillante.

Imports necesarios para proyecto.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
import os



In [ ]:
#Conguiracion del yaml
with open('../config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

print("Configuración cargada:")
print(yaml.dump(config, default_flow_style=False))

semilla = config["training"]["seed"]

#semillas
torch.manual_seed(semilla)
np.random.seed(semilla)

## 1. Carga y Preprocesamiento de Datos
Vamos a cargar el CSV y crear secuencias. Uso frecuencia de 50hz, es decir cada seghundo se captan 50 datos, cada dos segundos 100...

In [ ]:
def load_and_window_data(filepath, window_size=100, step=50):
    print(f"Cargando datos desde {filepath}...")
    df = pd.read_csv(filepath)
    
    # sacamos ejes x,y,z
    features = df[['accel_x', 'accel_y', 'accel_z']].values
    
    min_vals = features.min(axis=0)
    max_vals = features.max(axis=0)
    features_norm = (features - min_vals) / (max_vals - min_vals)
    
    windows = []
    for i in range(0, len(features_norm) - window_size, step):
        windows.append(features_norm[i:i + window_size])
        
    windows = np.array(windows)
    windows = np.transpose(windows, (0, 2, 1))
    print(f"Secuencias creadas: {windows.shape}")
    return windows, min_vals, max_vals

# configuraicon
data_path = config['data']['train_path']
w_size = config['data']['window_size']
step_size = config['data']['step']

X_train, min_v, max_v = load_and_window_data(data_path, window_size=w_size, step=step_size)

In [ ]:
class VibrationDataset(Dataset):
    def __init__(self, data):
        self.data = torch.FloatTensor(data)
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.data[idx]

train_dataset = VibrationDataset(X_train)
#carga batch
b_size = config['training']['batch_size']
train_loader = DataLoader(train_dataset, batch_size=b_size, shuffle=True)

## 2. Definición del Modelo: Convolutional Autoencoder 1D
Usaremos Conv1D porque captura excelentemente los patrones locales temporales (el "traqueteo").

In [ ]:
class Conv1DAutoencoder(nn.Module):
    def __init__(self, in_channels=3, filters=[16, 32, 64], kernel_size=5):
        super(Conv1DAutoencoder, self).__init__()
        
        # ENCODER
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, filters[0], kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.Conv1d(filters[0], filters[1], kernel_size=kernel_size, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv1d(filters[1], filters[2], kernel_size=kernel_size, stride=2, padding=2),
            nn.ReLU()
        )
        
        # DECODER
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(filters[2], filters[1], kernel_size=kernel_size, stride=2, padding=2, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(filters[1], filters[0], kernel_size=kernel_size, stride=2, padding=2, output_padding=1),
            nn.ReLU(),
        )
        
        self.final_conv = nn.Sequential(
             nn.ConvTranspose1d(filters[0], in_channels, kernel_size=7, stride=2, padding=3, output_padding=1),
             nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        decoded = self.final_conv(decoded)
        
        if decoded.size(2) != x.size(2):
             decoded = torch.nn.functional.interpolate(decoded, size=x.size(2))
             
        return decoded

# Inicializamos usando el YAML
model = Conv1DAutoencoder(
    in_channels=config['model']['in_channels'],
    filters=config['model']['encoder_filters'],
    kernel_size=config['model']['kernel_size']
)
dummy_input = torch.randn(10, config['model']['in_channels'], config['data']['window_size'])
output = model(dummy_input)
print(f"Shape de entrada: {dummy_input.shape}, Shape de salida: {output.shape}")

## 3. Bucle de Entrenamiento

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

model = model.to(device)
criterion = nn.MSELoss()

# LR y Epochs desde YAML
lr = config['training']['learning_rate']
epochs = config['training']['epochs']

optimizer = optim.Adam(model.parameters(), lr=lr)

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for batch_x, _ in train_loader:
        batch_x = batch_x.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_x)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_x.size(0)
        
    train_loss = train_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {train_loss:.6f}")

## 4. Visualización de la Reconstrucción
Vamos a ver cómo reconstruye una onda normal.

In [ ]:
model.eval()
sample = train_dataset[0][0].unsqueeze(0).to(device)
with torch.no_grad():
    reconstruction = model(sample)

# Pasamos a CPU y numpy para plotear el eje Z
orig_z = sample[0, 2, :].cpu().numpy()
recon_z = reconstruction[0, 2, :].cpu().numpy()

plt.figure(figsize=(10, 4))
plt.plot(orig_z, label="Original (Z)")
plt.plot(recon_z, label="Reconstruido (Z)", linestyle='--')
plt.title("Original vs Reconstrucción (Vía Normal)")
plt.legend()
plt.show()